<!-- field-paper-role-banner -->
> 🟩 **정답·해설본** — 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 수식과 tensor shape를 설명하세요.

# 지식 증류 · 모델 경량화 · 02. Attention Transfer

## Paying More Attention to Attention: Improving the Performance of Convolutional Neural Networks via Attention Transfer

- **원 논문:** [Paying More Attention to Attention: Improving the Performance of Convolutional Neural Networks via Attention Transfer](https://openreview.net/forum?id=Sks9_ajex)
- **저자 / 발표:** Sergey Zagoruyko, Nikos Komodakis · ICLR 2017 (2017)
- **난이도 / 예상 시간:** 중급 · 약 75분
- **선수 지식:** PyTorch tensor와 autograd, softmax·cross entropy, 선형대수 기초
- **로컬 학습 데이터:** `data/field_curriculum/compression_bench.npz`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** CNN feature를 spatial attention map으로 요약하고 teacher와 student의 정규화된 map을 맞춘다.

**축소하거나 생략한 부분:** 원 논문은 Wide ResNet teacher/student와 CIFAR/ImageNet을 사용한다. 실습은 16×16 합성 영상의 작은 CNN이다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §2.1, Eq. (1)–(2): activation attention | Task 1 | channel power 합 |
| §2.1, Eq. (3): vector normalization | Task 1 | scale invariance |
| §2.2, Eq. (4): attention transfer loss | Task 2 | teacher/student map distance |
| §3: classification + AT experiments | Task 3 | joint loss와 accuracy |

## 미니 재현
채널 수가 달라도 feature가 **어디를 보고 있는지**를 2차원 attention map으로 비교할 수 있습니다.

## 포트폴리오 해설: 정확도와 효율의 계약

$$\mathcal L_{AT}=\frac{\beta}{2}\sum_j\left\|\frac{Q_S^j}{\|Q_S^j\|_2}-\frac{Q_T^j}{\|Q_T^j
\|_2}\right\|_2^2$$

- **기호 정의:** Q는 channel activation을 합한 spatial attention map, β는 전달 강도입니다.
- **수식의 역할:** 채널 수가 다른 teacher/student도 어디를 보는지라는 공간 정보로 정렬합니다.
- **구현 이유:** 정확도만 남기면 경량화의 비용 절감 주장을 검증할 수 없으므로,
  핵심 변환과 품질·비용 측정을 독립 함수와 method로 분리했습니다.
- **Task/코드 대응:** Task P의 논문 전용 class와 `forward`, `compute_loss`,
  `training_step`, `fit_portfolio_model`, `evaluate_portfolio_model`이
  §2.1, Eq. (1)–(2): activation attention의 흐름을 명시합니다.
- **입출력 shape:** image [B, 1, 16, 16] → feature [B, C, H, W] → attention [B, H·W]
- **평가:** 정규화 attention MSE와 분류 accuracy
- **원문 대비 한계:** 원 논문은 Wide ResNet teacher/student와 CIFAR/ImageNet을 사용한다. 실습은 16×16 합성 영상의 작은 CNN이다.

경량화 실험에서는 정확도만 보지 않고 parameter·sparsity·bit·latency 중 논문이
주장한 비용을 함께 기록해야 합니다. 아래 코드는 단계별 중간값을 남깁니다.

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** feature [B,C,H,W] → attention map [B,H,W] → normalized [B,HW]
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

In [ ]:
from pathlib import Path
import math
import time
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import Tensor, nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(20260814)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
torch.set_default_device(DEVICE)
print(ACCELERATOR.summary())


def locate(relative):
    for base in (Path.cwd(), *Path.cwd().parents):
        path = base / relative
        if path.exists():
            return path
    raise FileNotFoundError(relative)


dataset_path = locate("data/field_curriculum/compression_bench.npz")
raw = np.load(dataset_path, allow_pickle=False)
features = torch.from_numpy(raw["features"]).float().to(DEVICE)
labels = torch.from_numpy(raw["labels"]).long().to(DEVICE)
teacher_logits = torch.from_numpy(raw["teacher_logits"]).float().to(DEVICE)
teacher_weights = torch.from_numpy(raw["teacher_weights"]).float().to(DEVICE)
calibration = torch.from_numpy(raw["calibration_activations"]).float().to(DEVICE)
dense_weights = torch.from_numpy(raw["dense_weights"]).float().to(DEVICE)
train_idx = torch.from_numpy(raw["train_idx"]).long().to(DEVICE)
test_idx = torch.from_numpy(raw["test_idx"]).long().to(DEVICE)
images = torch.from_numpy(raw["images"]).float().to(DEVICE)
image_labels = torch.from_numpy(raw["image_labels"]).long().to(DEVICE)
image_train_idx = torch.from_numpy(raw["image_train_idx"]).long().to(DEVICE)
image_test_idx = torch.from_numpy(raw["image_test_idx"]).long().to(DEVICE)
token_embeddings = torch.from_numpy(raw["token_embeddings"]).float().to(DEVICE)
teacher_hidden = torch.from_numpy(raw["teacher_hidden"]).float().to(DEVICE)
teacher_attention = torch.from_numpy(raw["teacher_attention"]).float().to(DEVICE)
candidate_configs = torch.from_numpy(raw["candidate_configs"]).float()
assert features.shape == (512, 32) and teacher_logits.shape == (512, 4)
assert images.shape == (320, 1, 16, 16)
assert teacher_hidden.shape == (256, 6, 24)
print(
    dataset_path.name, "features", tuple(features.shape), "images", tuple(images.shape)
)

### 구현 단계 2 · 핵심 연산·모델

- **원문 위치:** §2.1, Eq. (1)–(2): activation attention / §2.1, Eq. (3): vector normalization
- **이 셀의 책임:** Task 1
- **Tensor shape 계약:** feature [B,C,H,W] → attention map [B,H,W] → normalized [B,HW]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** channel power 합 / scale invariance. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def attention_map(feature, power=2):
    spatial = feature.abs().pow(power).sum(1).flatten(1)
    return F.normalize(spatial, p=2, dim=1)


probe = torch.rand(4, 5, 8, 8)
assert attention_map(probe).shape == (4, 64)
assert torch.allclose(attention_map(probe), attention_map(probe * 7), atol=1e-5)

### 구현 단계 3 · 목적함수·학습 update

- **원문 위치:** §2.2, Eq. (4): attention transfer loss
- **이 셀의 책임:** Task 2
- **Tensor shape 계약:** feature [B,C,H,W] → attention map [B,H,W] → normalized [B,HW]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** teacher/student map distance. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
smooth = F.avg_pool2d(images, 3, 1, 1)
edge_x = F.pad(images[:, :, :, 1:] - images[:, :, :, :-1], (0, 1, 0, 0))
teacher_feature = torch.cat([images, smooth, edge_x], 1)


class ATStudent(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(1, 6, 3, padding=1)
        self.head = nn.Linear(6 * 16 * 16, 4)

    def forward(self, x):
        feature = F.relu(self.conv(x))
        return self.head(feature.flatten(1)), feature


student = ATStudent()
_, student_feature = student(images[:8])
at_loss = F.mse_loss(attention_map(student_feature), attention_map(teacher_feature[:8]))
assert torch.isfinite(at_loss) and at_loss > 0

### 구현 단계 4 · 평가·완료 증거

- **원문 위치:** §2.2, Eq. (4): attention transfer loss / §3: classification + AT experiments
- **이 셀의 책임:** Task 2 / Task 3
- **Tensor shape 계약:** feature [B,C,H,W] → attention map [B,H,W] → normalized [B,HW]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** teacher/student map distance / joint loss와 accuracy. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
optimizer = torch.optim.Adam(student.parameters(), lr=0.02)
losses = []
attentions = []
for _ in range(45):
    optimizer.zero_grad()
    logits, feature = student(images[image_train_idx])
    ce = F.cross_entropy(logits, image_labels[image_train_idx])
    at = F.mse_loss(
        attention_map(feature), attention_map(teacher_feature[image_train_idx])
    )
    loss = ce + 25 * at
    loss.backward()
    optimizer.step()
    losses.append(float(ce.detach()))
    attentions.append(float(at.detach()))
with torch.no_grad():
    accuracy = (
        (student(images[image_test_idx])[0].argmax(-1) == image_labels[image_test_idx])
        .float()
        .mean()
        .item()
    )
assert losses[-1] < losses[0] and attentions[-1] < attentions[0] and accuracy > 0.7
plt.plot(losses, label="CE")
plt.plot(attentions, label="AT")
plt.legend()
plt.show()
print(f"accuracy={accuracy:.1%}")

### Task P1 · 압축 대상과 핵심 모델

tensor shape와 비용 단위를 먼저 확인한 뒤 논문 전용 class의 `forward`와
`compute_loss`에 정확도·표현 보존 계약을 명시합니다.

### 구현 단계 5 · 핵심 연산·모델

- **원문 위치:** §2.1, Eq. (1)–(2): activation attention / §2.1, Eq. (3): vector normalization / §2.2, Eq. (4): attention transfer loss / §3: classification + AT experiments
- **이 셀의 책임:** Task 1 / Task 2 / Task 3
- **Tensor shape 계약:** feature [B,C,H,W] → attention map [B,H,W] → normalized [B,HW]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** channel power 합 / scale invariance / teacher/student map distance / joint loss와 accuracy. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class PortfolioConfig:
    hidden_channels: int = 8
    teacher_channels: int = 16
    attention_weight: float = 2.0
    learning_rate: float = 0.02
    teacher_steps: int = 8
    steps: int = 16


def prepare_batch() -> tuple[Tensor, Tensor]:
    batch = images[image_train_idx[:128]]
    target = image_labels[image_train_idx[:128]]
    return batch, target


class AttentionTeacher(nn.Module):
    def __init__(self, channels: int):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, channels, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Linear(channels, 4)
        self.requires_grad_(False)

    def forward(self, inputs: Tensor) -> tuple[Tensor, Tensor]:
        feature_map = self.features(inputs)
        logits = self.classifier(feature_map.mean(dim=(2, 3)))
        return logits, feature_map


class AttentionTransferPortfolioModel(nn.Module):
    def __init__(self, config: PortfolioConfig):
        super().__init__()
        self.config = config
        self.teacher = AttentionTeacher(config.teacher_channels)
        self.features = nn.Sequential(
            nn.Conv2d(1, config.hidden_channels, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Linear(config.hidden_channels, 4)

    @staticmethod
    def spatial_attention(feature_map: Tensor) -> Tensor:
        attention = feature_map.square().sum(dim=1).flatten(1)
        return F.normalize(attention, dim=1)

    def forward(self, inputs: Tensor) -> tuple[Tensor, Tensor, Tensor]:
        feature_map = self.features(inputs)
        logits = self.classifier(feature_map.mean(dim=(2, 3)))
        student_map = self.spatial_attention(feature_map)
        with torch.no_grad():
            _, teacher_feature = self.teacher(inputs)
            teacher_map = self.spatial_attention(teacher_feature)
        return logits, student_map, teacher_map

    def compute_loss(
        self,
        outputs: tuple[Tensor, Tensor, Tensor],
        targets: Tensor,
    ) -> Tensor:
        logits, student_map, teacher_map = outputs
        classification = F.cross_entropy(logits, targets)
        transfer = F.mse_loss(student_map, teacher_map)
        return classification + self.config.attention_weight * transfer

    def training_step(
        self,
        inputs: Tensor,
        targets: Tensor,
    ) -> Tensor:
        return self.compute_loss(self(inputs), targets)

### Task P2 · 학습·pruning·distillation update

`fit_portfolio_model`에서 optimizer, mask/EMA/quantization 경계와 update 순서를
드러냅니다. 비용 절감이 학습 경로에 미치는 영향을 함께 읽으세요.

### 구현 단계 6 · 목적함수·학습 update

- **원문 위치:** §2.2, Eq. (4): attention transfer loss / §3: classification + AT experiments
- **이 셀의 책임:** Task 2 / Task 3
- **Tensor shape 계약:** feature [B,C,H,W] → attention map [B,H,W] → normalized [B,HW]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** teacher/student map distance / joint loss와 accuracy. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def fit_portfolio_model(model: AttentionTransferPortfolioModel) -> list[float]:
    inputs, targets = prepare_batch()
    model.teacher.requires_grad_(True)
    teacher_optimizer = torch.optim.Adam(
        model.teacher.parameters(),
        lr=model.config.learning_rate,
    )
    history = []
    for _ in range(model.config.teacher_steps):
        teacher_logits_local, _ = model.teacher(inputs)
        teacher_loss = F.cross_entropy(teacher_logits_local, targets)
        teacher_optimizer.zero_grad()
        teacher_loss.backward()
        teacher_optimizer.step()
        history.append(float(teacher_loss.detach().cpu()))
    model.teacher.requires_grad_(False)
    frozen_before = [parameter.clone() for parameter in model.teacher.parameters()]
    trainable = [
        parameter for parameter in model.parameters() if parameter.requires_grad
    ]
    optimizer = torch.optim.Adam(trainable, lr=model.config.learning_rate)
    for _ in range(model.config.steps):
        loss = model.training_step(inputs, targets)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        history.append(float(loss.detach().cpu()))
    frozen_after = list(model.teacher.parameters())
    assert all(
        torch.equal(before, after) for before, after in zip(frozen_before, frozen_after)
    )
    return history

### Task P3 · 품질과 비용을 함께 평가

accuracy 또는 reconstruction 품질과 parameter·sparsity·bit·latency를 별도로
기록합니다. 로컬 지표는 원문 하드웨어 benchmark의 대체값이 아닙니다.

### 구현 단계 7 · 평가·완료 증거

- **원문 위치:** §3: classification + AT experiments
- **이 셀의 책임:** Task 3
- **Tensor shape 계약:** feature [B,C,H,W] → attention map [B,H,W] → normalized [B,HW]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** joint loss와 accuracy. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def evaluate_portfolio_model(
    model: AttentionTransferPortfolioModel,
) -> dict[str, float]:
    with torch.no_grad():
        logits, student_map, teacher_map = model(images[image_test_idx])
        accuracy = (logits.argmax(1) == image_labels[image_test_idx]).float().mean()
        transfer_error = F.mse_loss(student_map, teacher_map)
    teacher_trainable = sum(
        parameter.requires_grad for parameter in model.teacher.parameters()
    )
    return {
        "test_accuracy": float(accuracy.cpu()),
        "teacher_student_attention_mse": float(transfer_error.cpu()),
        "teacher_trainable_tensors": float(teacher_trainable),
    }


portfolio_model = AttentionTransferPortfolioModel(PortfolioConfig())
portfolio_history = fit_portfolio_model(portfolio_model)
portfolio_metrics = evaluate_portfolio_model(portfolio_model)
assert np.isfinite(portfolio_history).all()
print(portfolio_metrics)

## 정답 해석 가이드

핵심은 최종 숫자를 외우는 것이 아니라 **원문 위치 → 수식 → 코드 → 반증 가능한 증거**를
연결하는 것입니다. 이 노트북은 다음 원 규모 조건을 재현하지 않았습니다:

> 원 논문은 Wide ResNet teacher/student와 CIFAR/ImageNet을 사용한다. 실습은 16×16 합성 영상의 작은 CNN이다.

다음 실험에서는 데이터 크기, 모델 폭, 학습 step 중 하나만 바꾸고 동일 seed에서 비교하세요.